# Score SDE — Score-Based Generative Modeling through SDEs (2020)

_Papers · Generative Models_

**Cast diffusion as a continuous stochastic differential equation: a forward SDE turns data into noise, and a reverse SDE driven by the learned score (the gradient of log-density) turns noise back into data.**

---

This notebook is a practice scaffold for the **Score SDE — Score-Based Generative Modeling through SDEs (2020)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, math
torch.manual_seed(0)

# --- 0. Worked example: the score of a Gaussian (the reverse SDE's only learned term). ---
# For x ~ N(mu, sigma^2 I): grad_x log p(x) = -(x - mu)/sigma^2
mu, sigma, x = torch.tensor([1.0, -0.5]), 0.5, torch.tensor([1.4, 0.3])
score = -(x - mu) / sigma**2
print("worked score:", [round(v, 4) for v in score.tolist()])          # [-1.6, -3.2]
# Denoising-score-matching target is the SAME vector: for x_t = x0 + sigma_t*z, score = -z/sigma_t
x0d, z, st = torch.tensor([1.0, -0.5]), torch.tensor([0.8, 1.6]), 0.5
print("DSM target -z/sigma_t:", [round(v, 4) for v in (-z / st).tolist()])   # [-1.6, -3.2]

# --- 1. The VE SDE noise schedule (geometric sigma_t, t in [0,1]). ---
sigma_min, sigma_max = 0.05, 3.0
log_ratio = math.log(sigma_max / sigma_min)
def sigma_at(t):                                    # standard deviation added at time t
    return sigma_min * (sigma_max / sigma_min) ** t

# --- 2. A 2-D toy distribution: 8 Gaussian clusters on a ring (8 modes). ---
def sample_data(n):
    k = torch.randint(0, 8, (n,))
    ang = k.float() / 8 * 2 * math.pi
    return torch.stack([torch.cos(ang), torch.sin(ang)], 1) * 2.0 + torch.randn(n, 2) * 0.1

# --- 3. The score network s_theta(x, t): a small MLP. use_t toggles the ablation. ---
class ScoreNet(nn.Module):
    def __init__(self, h=128, use_t=True):
        super().__init__(); self.use_t = use_t
        self.net = nn.Sequential(nn.Linear(2 + 1, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, 2))
    def forward(self, x, t):
        te = t.unsqueeze(1)
        if not self.use_t:
            te = torch.zeros_like(te)               # ablation: hide the time
        return self.net(torch.cat([x, te], 1))

# --- 4. Train with denoising score matching (Eq. 7), lambda(t) = sigma_t^2. ---
def train(use_t=True, steps=4000):
    torch.manual_seed(0)
    net = ScoreNet(use_t=use_t)
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    for step in range(steps):
        x0 = sample_data(512)
        t  = torch.rand(512)                        # continuous time in [0,1]
        s  = sigma_at(t).unsqueeze(1)
        z  = torch.randn_like(x0)
        xt = x0 + s * z                             # VE perturbation: x_t = x0 + sigma_t * z
        target = -z / s                             # known score of the Gaussian kernel
        loss = ((s * (net(xt, t) - target)) ** 2).sum(1).mean()   # Eq. 7 with lambda = sigma^2
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 1000 == 0:
            print(f"  step {step:4d}  loss {loss.item():.4f}")
    return net

# --- 5. Sample by integrating the reverse SDE (Eq. 6) with Euler-Maruyama. ---
@torch.no_grad()
def sample(net, n=500, N=400, snapshot_at=(0.99, 0.5, 0.0)):
    x  = torch.randn(n, 2) * sigma_max              # x(1) ~ N(0, sigma_max^2 I)
    ts = torch.linspace(1.0, 1e-3, N)
    dt = ts[1] - ts[0]                              # negative: time runs backward
    snaps = {}
    for i in range(N):
        t  = ts[i]
        s  = sigma_at(t)
        g2 = 2 * log_ratio * s ** 2                 # g(t)^2 = d[sigma^2]/dt for geometric sigma
        sc = net(x, torch.full((n,), float(t)))     # learned score s_theta(x, t)
        x  = x - g2 * sc * dt                        # reverse drift: -g^2 * score * dt   (dt<0)
        if i < N - 1:
            x = x + math.sqrt(float(g2 * (-dt))) * torch.randn_like(x)   # g(t) dW-bar
        for key in snapshot_at:
            if abs(float(t) - key) < abs(float(dt)) / 2 and key not in snaps:
                snaps[key] = x.clone()
    snaps[0.0] = x.clone()
    return x, snaps

print("TRAIN (with time):")
net = train(use_t=True)
samples, snaps = sample(net)

# fraction of samples landing on the ring of 8 clusters
ang   = torch.arange(8).float() / 8 * 2 * math.pi
modes = torch.stack([torch.cos(ang), torch.sin(ang)], 1) * 2.0
d = torch.cdist(samples, modes).min(1).values
print("\nfrac of samples within 0.4 of a cluster:", round((d < 0.4).float().mean().item(), 3))
print("mean radius (should approach ring radius 2.0):", round(samples.norm(dim=1).mean().item(), 3))
# Typical: ~0.99 of samples land on a cluster; mean radius ~1.99. (Our small run -- not the paper's number.)

# --- 6. ABLATION: hide the time input, retrain, resample. Samples scatter. ---
print("\nABLATION (no time input):")
net_no_t = train(use_t=False)
samp_no_t, _ = sample(net_no_t)
d2 = torch.cdist(samp_no_t, modes).min(1).values
print("frac within 0.4 of a cluster, NO time:", round((d2 < 0.4).float().mean().item(), 3))
# Much lower (~0.15): one network cannot give the right-magnitude score across all noise levels.

## Visualize the data & results

_Starting from pure VE noise, does integrating the reverse SDE (driven by the learned score) pull the cloud back onto the 8 data clusters?_

In [ ]:
import torch, torch.nn as nn, math
torch.manual_seed(0)

sigma_min, sigma_max = 0.05, 3.0
log_ratio = math.log(sigma_max / sigma_min)
def sigma_at(t): return sigma_min * (sigma_max / sigma_min) ** t

def sample_data(n):
    k = torch.randint(0, 8, (n,)); ang = k.float() / 8 * 2 * math.pi
    return torch.stack([torch.cos(ang), torch.sin(ang)], 1) * 2.0 + torch.randn(n, 2) * 0.1

class ScoreNet(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, x, t): return self.net(torch.cat([x, t.unsqueeze(1)], 1))

net = ScoreNet(); opt = torch.optim.Adam(net.parameters(), lr=2e-3)
for _ in range(4000):                                # denoising score matching (Eq. 7)
    x0 = sample_data(512); t = torch.rand(512); s = sigma_at(t).unsqueeze(1)
    z = torch.randn_like(x0); xt = x0 + s * z
    loss = ((s * (net(xt, t) - (-z / s))) ** 2).sum(1).mean()
    opt.zero_grad(); loss.backward(); opt.step()

@torch.no_grad()                                     # reverse SDE (Eq. 6), Euler-Maruyama
def sample(n=80, N=400, snap=(0.99, 0.5, 0.0)):
    x = torch.randn(n, 2) * sigma_max
    ts = torch.linspace(1.0, 1e-3, N); dt = ts[1] - ts[0]; snaps = {}
    for i in range(N):
        t = ts[i]; s = sigma_at(t); g2 = 2 * log_ratio * s ** 2
        sc = net(x, torch.full((n,), float(t)))
        x = x - g2 * sc * dt
        if i < N - 1: x = x + math.sqrt(float(g2 * (-dt))) * torch.randn_like(x)
        for k in snap:
            if abs(float(t) - k) < abs(float(dt)) / 2 and k not in snaps: snaps[k] = x[:60].clone()
    snaps[0.0] = x[:60].clone(); return snaps

torch.manual_seed(7); snaps = sample()
for k in (0.99, 0.5, 0.0):
    print(f"t={k:.2f}", [[round(p[0].item(), 3), round(p[1].item(), 3)] for p in snaps[k][:5]], "...")
# t=0.99 -> exploded blob; t=0.5 -> pulling toward the ring; t=0.0 -> seated on the 8 clusters.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compute the score of a 2-D Gaussian $\mathcal{N}(\mu,\sigma^2 I)$ with $\mu=(0,2)$, $\sigma=1$, at $x=(1,0)$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use $\nabla_x \log p(x) = -(x-\mu)/\sigma^2$. — _This is the score of an isotropic Gaussian (derived in the worked example)._
- $x-\mu=(1-0,\,0-2)=(1,-2)$; $\sigma^2=1$. — _Subtract the mean componentwise._
- Negate and divide by $\sigma^2$: $-(1,-2)/1=(-1,2)$. — _The score points back toward the mean $\mu$._

**Answer:** The score is $(-1,\,2)$ — pointing from $x=(1,0)$ toward $\mu=(0,2)$.

</details>

**Problem 2.** Why does the reverse SDE need the score but the forward SDE does not?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Look at Eq. 5 vs Eq. 6. — _The forward SDE has only $f$ and $g$, both chosen by hand._
- Eq. 6 adds the term $-g(t)^2\nabla_x\log p_t(x)$. — _Anderson's reversal theorem (§3.2) introduces the score as the extra force when running time backward._

**Answer:** Noising is a fixed recipe, so the forward SDE needs no data information. Reversing it requires knowing where data density is — that information IS the score, the one learned term in Eq. 6.

</details>

**Problem 3.** ABLATION: what happens to samples if you hide the time input $t$ from $s_\theta$, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the score scales like $-z/\sigma_t$, and $\sigma_t$ varies hugely with $t$. — _Early in sampling the noise is large; near the end it is tiny — the correct score differs by orders of magnitude._
- Without $t$, one network must output a single score field for all noise levels. — _It can no longer tell a barely-noised point from a wildly-noised one, so it applies the wrong-magnitude correction._
- Run both and compare the fraction of samples landing on a cluster. — _In our run it drops from ~0.99 (with $t$) to ~0.15 (no $t$)._

**Answer:** Samples scatter and miss the clusters. The fraction within 0.4 of a cluster falls from ≈0.99 to ≈0.15 in our small run (not the paper's number) — confirming the time-conditioning is essential.

</details>